# Python for Agentic AI — Runnable Notebook

> All 6 concepts with executable code.
> **Rule:** Predict the output before running each cell.

Concepts covered:
1. Decorators
2. Dataclasses
3. Typing Basics
4. Enum
5. `@property` and `__call__`
6. Pydantic v2

---
## 1. Decorators

**Core idea:** A decorator is a function that takes a function and returns a new function. `@` is shorthand for `func = decorator(func)`.

### 1.1 — Functions are first-class objects

In [ ]:
def greet():
    print("Hello")

# Assign to a variable
x = greet
x()   # same as greet()

# Pass to another function
def execute(func):
    print("Calling function:")
    func()

execute(greet)

### 1.2 — Build a decorator manually (without @)

In [ ]:
def my_decorator(func):
    def wrapper():
        print("-- Before --")
        func()
        print("-- After --")
    return wrapper  # return the wrapper, don't call it

def say_hi():
    print("Hi!")

# Manual application — this is what @ does
say_hi = my_decorator(say_hi)
say_hi()

### 1.3 — Using @ syntax (syntactic sugar)

In [ ]:
def my_decorator(func):
    def wrapper():
        print("-- Before --")
        func()
        print("-- After --")
    return wrapper

@my_decorator   # equivalent to: say_hello = my_decorator(say_hello)
def say_hello():
    print("Hello!")

say_hello()

### 1.4 — The problem without `functools.wraps`

In [ ]:
def broken_decorator(func):
    def wrapper():
        return func()
    return wrapper

@broken_decorator
def important_function():
    """This docstring matters."""
    pass

# Metadata is lost — greet is now wrapper
print(f"Name: {important_function.__name__}")
print(f"Doc: {important_function.__doc__}")

### 1.5 — Fixed with `functools.wraps`

In [ ]:
from functools import wraps

def good_decorator(func):
    @wraps(func)   # copies __name__, __doc__, __module__ etc.
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@good_decorator
def important_function():
    """This docstring matters."""
    pass

print(f"Name: {important_function.__name__}")
print(f"Doc: {important_function.__doc__}")

### 1.6 — Generic decorator with *args/**kwargs

In [ ]:
from functools import wraps
import time

def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper

@timer
def add(a, b):
    return a + b

@timer
def greet(name, greeting="Hello"):
    return f"{greeting}, {name}!"

print(add(3, 4))
print(greet("Tosho", greeting="Hey"))

### 1.7 — Decorator factory (decorator with arguments)

In [ ]:
from functools import wraps

def logger(level="INFO"):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            print(f"[{level}] Calling: {func.__name__}")
            result = func(*args, **kwargs)
            print(f"[{level}] Done: {func.__name__}")
            return result
        return wrapper
    return decorator

# @logger("INFO") → logger("INFO") returns decorator → decorator wraps the function
@logger("DEBUG")
def search(query):
    return f"Results for: {query}"

@logger("INFO")
def summarize(text):
    return text[:50]

print(search("pydantic tutorial"))
print(summarize("Pydantic is a data validation library..."))

### 1.8 — Stacking decorators

In [ ]:
from functools import wraps

def bold(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return f"**{func(*args, **kwargs)}**"
    return wrapper

def uppercase(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs).upper()
    return wrapper

# Applied bottom-up: uppercase first, then bold
@bold
@uppercase
def greet(name):
    return f"hello {name}"

print(greet("world"))   # predict before running!

---
## 2. Dataclasses

**Core idea:** `@dataclass` auto-generates `__init__`, `__repr__`, `__eq__` — eliminating boilerplate for data-holding classes.

### 2.1 — The problem (manual boilerplate)

In [ ]:
# Without dataclass — all manual
class ToolResult:
    def __init__(self, tool_name, output, success):
        self.tool_name = tool_name
        self.output = output
        self.success = success

    def __repr__(self):
        return f"ToolResult(tool_name={self.tool_name!r}, output={self.output!r}, success={self.success!r})"

    def __eq__(self, other):
        return (self.tool_name == other.tool_name and
                self.output == other.output and
                self.success == other.success)

r1 = ToolResult("search", "Found 10 results", True)
r2 = ToolResult("search", "Found 10 results", True)
print(r1)
print(r1 == r2)

### 2.2 — Same class with @dataclass

In [ ]:
from dataclasses import dataclass

@dataclass
class ToolResult:
    tool_name: str
    output: str
    success: bool

r1 = ToolResult("search", "Found 10 results", True)
r2 = ToolResult("search", "Found 10 results", True)
print(r1)          # __repr__ auto-generated
print(r1 == r2)    # __eq__ compares field values

### 2.3 — Default values

In [ ]:
from dataclasses import dataclass

@dataclass
class Message:
    role: str
    content: str
    model: str = "gpt-5"      # default
    tokens: int = 0            # default

m1 = Message("user", "What is Python?")
m2 = Message("assistant", "A programming language", tokens=42)

print(m1)
print(m2)

### 2.4 — The mutable default trap

In [ ]:
from dataclasses import dataclass

# Mutable default not allowed directly in dataclasses
try:
    @dataclass
    class BadAgent:
        tools: list = []   # Error!
except Exception as e:
    print(f"Error ({type(e).__name__}): {e}")

# Python prevents mutable defaults to avoid shared state bugs
print("Use field(default_factory=list) instead.")


### 2.5 — Correct: field(default_factory=...)

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Agent:
    name: str
    tools: list = field(default_factory=list)   # each instance gets its own list
    metadata: dict = field(default_factory=dict)

a1 = Agent("ResearchBot")
a2 = Agent("CodeBot")

a1.tools.append("search")
a2.tools.append("code_exec")

print(f"a1 tools: {a1.tools}")
print(f"a2 tools: {a2.tools}")   # not affected by a1

### 2.6 — field() options

In [ ]:
from dataclasses import dataclass, field

@dataclass
class User:
    name: str
    age: int
    password: str = field(repr=False)       # hidden from print
    temp_id: int = field(compare=False)     # excluded from ==
    score: float = field(init=False)        # not in __init__, set in __post_init__

    def __post_init__(self):
        self.score = self.age * 0.5   # computed after init

u1 = User("Alice", 30, "secret123", temp_id=999)
u2 = User("Alice", 30, "different_pw", temp_id=888)

print(u1)          # password hidden
print(u1 == u2)    # True — temp_id excluded from comparison
print(u1.score)    # computed in __post_init__

### 2.7 — frozen=True

In [ ]:
from dataclasses import dataclass, field

@dataclass(frozen=True)
class AgentConfig:
    model: str
    temperature: float
    max_tokens: int
    tools: tuple = field(default_factory=tuple)

cfg = AgentConfig("gpt-5", 0.7, 1000)
print(cfg)

# Try to modify — should raise FrozenInstanceError
try:
    cfg.model = "gpt-4"
except Exception as e:
    print(f"Blocked: {type(e).__name__}: {e}")

# frozen=True enables hashing
config_cache = {cfg: "cached_result"}
print(f"Used as dict key: {config_cache[cfg]}")

---
## 3. Typing Basics

**Core idea:** Modern Python 3.10+ uses `X | Y`, `list[X]`, `dict[K,V]` directly. Only `Any`, `Callable`, `Literal` still need `from typing import`.

In [ ]:
from typing import Any, Callable, Literal

# Modern syntax (Python 3.10+)
def process_agent(
    name: str,
    api_key: str | None,                     # Optional
    tools: list[str],                         # List
    metadata: dict[str, Any],                 # Dict with Any values
    callback: Callable[[str], None],          # Callable
    mode: Literal["chat", "tool", "planner"], # Fixed values
) -> dict[str, Any]:
    return {"name": name, "mode": mode}

# Type hints are documentation + Pydantic/mypy input
# They don't affect runtime behavior in plain Python
def add(a: int, b: int) -> int:
    return a + b

print(add("hello", "world"))   # still works — Python doesn't enforce at runtime

In [ ]:
# type[X] — the class itself, not an instance
class BaseAgent:
    def run(self): pass

class ResearchAgent(BaseAgent):
    def run(self): print("Researching...")

class CodeAgent(BaseAgent):
    def run(self): print("Coding...")

def create_agent(cls: type[BaseAgent]) -> BaseAgent:
    return cls()

agent = create_agent(ResearchAgent)
agent.run()

agent2 = create_agent(CodeAgent)
agent2.run()

---
## 4. Enum

**Core idea:** Define a fixed, safe set of constants. Prevents magic strings and magic numbers from causing silent runtime bugs.

### 4.1 — The problem with magic strings

In [ ]:
# Without Enum — anything is valid
def run_agent(mode: str):
    if mode == "chat":
        print("Running in chat mode")
    elif mode == "tool":
        print("Running in tool mode")
    else:
        print("Unknown mode — silent failure")

run_agent("chat")
run_agent("Chat")   # typo — no error, falls through
run_agent("chatt")  # typo — no error, falls through

### 4.2 — Basic Enum

In [ ]:
from enum import Enum

class AgentMode(Enum):
    CHAT = "chat"
    TOOL = "tool"
    PLANNER = "planner"

mode = AgentMode.CHAT
print(mode)          # AgentMode.CHAT
print(mode.name)     # CHAT
print(mode.value)    # chat
print(type(mode))    # <enum 'AgentMode'>

### 4.3 — IntEnum and auto()

In [ ]:
from enum import Enum, IntEnum, auto

class Priority(IntEnum):
    LOW = 1
    MEDIUM = 2
    HIGH = 3

# Members compare like integers
print(Priority.HIGH > Priority.LOW)   # True
print(Priority.HIGH == 3)             # True
print(sorted([Priority.MEDIUM, Priority.HIGH, Priority.LOW]))

class Status(Enum):
    IDLE = auto()
    RUNNING = auto()
    FINISHED = auto()
    ERROR = auto()

# auto() assigns 1, 2, 3, 4 automatically
for s in Status:
    print(f"{s.name}: {s.value}")

### 4.4 — Lookup by value and name

In [ ]:
from enum import Enum

class Provider(Enum):
    OPENAI = "openai"
    ANTHROPIC = "anthropic"
    GOOGLE = "google"

# Lookup by value
p = Provider("anthropic")
print(p)   # Provider.ANTHROPIC

# Lookup by name
p2 = Provider["OPENAI"]
print(p2)  # Provider.OPENAI

# Invalid value
try:
    Provider("unknown")
except ValueError as e:
    print(f"Error: {e}")

### 4.5 — Enum vs Literal

In [ ]:
from enum import Enum
from typing import Literal

# Use Enum when: shared across modules, need iteration, need runtime validation
class AgentState(Enum):
    IDLE = "idle"
    RUNNING = "running"
    FAILED = "failed"

# Iteration — not possible with Literal
for state in AgentState:
    print(state.name)

# Runtime validation — not possible with Literal
state = AgentState("running")
print(f"Valid state: {state}")

# Use Literal when: type hint only, used in one place
def set_mode(mode: Literal["fast", "slow"]) -> None:
    print(f"Mode set to {mode}")

set_mode("fast")

---
## 5. `@property` and `__call__`

**Core idea:** `@property` hides a method behind attribute syntax. `__call__` makes an object callable like a function.

### 5.1 — The problem @property solves

In [ ]:
# Without @property — no validation possible without breaking API
class BadUser:
    def __init__(self, age):
        self.age = age

u = BadUser(25)
u.age = -10   # nothing stops this
print(u.age)  # -10

### 5.2 — @property with getter, setter, deleter

In [ ]:
class User:
    def __init__(self, age: int):
        self._age = age   # private storage

    @property
    def age(self) -> int:
        print("  [getter called]")
        return self._age

    @age.setter
    def age(self, value: int):
        print(f"  [setter called with {value}]")
        if value < 0:
            raise ValueError("Age cannot be negative")
        self._age = value

    @age.deleter
    def age(self):
        print("  [deleter called]")
        del self._age

u = User(25)
print(u.age)      # calls getter
u.age = 30        # calls setter
print(u.age)

try:
    u.age = -5    # setter raises ValueError
except ValueError as e:
    print(f"Blocked: {e}")

### 5.3 — Computed property

In [ ]:
class AgentConfig:
    def __init__(self, model: str, max_tokens: int):
        self.model = model
        self.max_tokens = max_tokens

    @property
    def summary(self) -> str:
        return f"{self.model} (max_tokens={self.max_tokens})"

    @property
    def is_high_capacity(self) -> bool:
        return self.max_tokens > 2000

cfg = AgentConfig("gpt-5", 4096)
print(cfg.summary)            # computed on access, no ()
print(cfg.is_high_capacity)   # True

### 5.4 — `__call__` — making an object callable

In [ ]:
class Greeter:
    def __init__(self, greeting: str):
        self.greeting = greeting

    def __call__(self, name: str) -> str:
        return f"{self.greeting}, {name}!"

hello = Greeter("Hello")
hey = Greeter("Hey")

print(hello("Tosho"))   # Hello, Tosho!
print(hey("World"))     # Hey, World!
print(callable(hello))  # True — has __call__

### 5.5 — Stateful callable: SimilarityScorer

In [ ]:
class SimilarityScorer:
    """Callable object that carries threshold state."""

    def __init__(self, threshold: float):
        self.threshold = threshold
        self._call_count = 0

    def __call__(self, score: float) -> bool:
        self._call_count += 1
        return score >= self.threshold

    @property
    def call_count(self) -> int:
        return self._call_count

scorer = SimilarityScorer(0.8)

scores = [0.92, 0.65, 0.88, 0.72, 0.95]
results = [scorer(s) for s in scores]

print(f"Scores: {scores}")
print(f"Above threshold: {results}")
print(f"Total calls: {scorer.call_count}")

### 5.6 — Agentic AI: callable tool

In [ ]:
class SearchTool:
    """A tool that behaves like a function but holds state."""

    def __init__(self, name: str, max_results: int = 5):
        self.name = name
        self.max_results = max_results
        self._history: list[str] = []

    def __call__(self, query: str) -> str:
        self._history.append(query)
        return f"[{self.name}] Top {self.max_results} results for: '{query}'"

    @property
    def query_count(self) -> int:
        return len(self._history)

search = SearchTool("WebSearch", max_results=3)
print(search("pydantic validation"))
print(search("langchain agents"))
print(f"Queries made: {search.query_count}")

---
## 6. Pydantic v2

**Core idea:** Pydantic is the boundary between untrusted external data and your clean Python objects. Type hints become a runtime validation schema.

In [ ]:
# Make sure pydantic is installed
try:
    import pydantic
    print(f"Pydantic version: {pydantic.__version__}")
except ImportError:
    print("Run: pip install pydantic")

### 6.1 — Dataclass vs BaseModel

In [ ]:
from dataclasses import dataclass
from pydantic import BaseModel

@dataclass
class DataclassConfig:
    model: str
    temperature: float

# Dataclass trusts you — no validation
dc = DataclassConfig("gpt-4", "0.8")  # string passed, string stored
print(f"Dataclass temperature: {dc.temperature!r} (type: {type(dc.temperature).__name__})")

class PydanticConfig(BaseModel):
    model: str
    temperature: float

# Pydantic validates and converts
pc = PydanticConfig(model="gpt-4", temperature="0.8")  # string → float
print(f"Pydantic temperature: {pc.temperature!r} (type: {type(pc.temperature).__name__})")

### 6.2 — Validation errors

In [ ]:
from pydantic import BaseModel

class AgentConfig(BaseModel):
    model: str
    temperature: float
    max_tokens: int

# Valid
cfg = AgentConfig(model="gpt-5", temperature="0.7", max_tokens="1000")
print(cfg)

# Invalid
try:
    AgentConfig(model="gpt-5", temperature="high", max_tokens=1000)
except Exception as e:
    print("ValidationError:")
    print(e)


### 6.3 — Field() for constraints, defaults, descriptions

In [ ]:
from pydantic import BaseModel, Field

class AgentConfig(BaseModel):
    model: str = Field(default="gpt-4", min_length=3)
    temperature: float = Field(ge=0, le=2, description="Sampling temperature")
    max_tokens: int = Field(gt=0, le=4096)
    retries: int = Field(default=3, ge=1, le=10)

cfg = AgentConfig(model="gpt-5", temperature=0.7, max_tokens=1000)
print(cfg)

try:
    AgentConfig(model="gpt-5", temperature=5.0, max_tokens=1000)
except Exception as e:
    print("Blocked: temperature=5.0 exceeds max of 2.0")


### 6.4 — Field aliases (for camelCase JSON)

In [ ]:
from pydantic import BaseModel, Field

class ToolCall(BaseModel):
    tool_name: str = Field(alias="toolName")
    max_retries: int = Field(alias="maxRetries", default=3)

# Incoming JSON uses camelCase
tc = ToolCall(toolName="search", maxRetries=5)
print(tc.tool_name)     # snake_case access in Python
print(tc.max_retries)

### 6.5 — Nested models

In [ ]:
from pydantic import BaseModel

class Tool(BaseModel):
    name: str
    description: str

class Agent(BaseModel):
    name: str
    model: str
    tool: Tool   # nested model

# Pass a dict — Pydantic converts it to Tool automatically
agent = Agent(
    name="ResearchBot",
    model="gpt-5",
    tool={"name": "search", "description": "Search the web"}
)

print(agent)
print(agent.tool.name)   # access nested field
print(type(agent.tool))  # Tool, not dict

### 6.6 — Lists of models

In [ ]:
from pydantic import BaseModel

class Tool(BaseModel):
    name: str
    description: str

class Agent(BaseModel):
    name: str
    tools: list[Tool]   # list of nested models

agent = Agent(
    name="MultiToolAgent",
    tools=[
        {"name": "search", "description": "Search the web"},
        {"name": "calculator", "description": "Solve math"},
        {"name": "browser", "description": "Browse URLs"},
    ]
)

print(f"Agent has {len(agent.tools)} tools")
for tool in agent.tools:
    print(f"  - {tool.name}: {tool.description}")

### 6.7 — model_dump() and model_dump_json()

In [ ]:
from pydantic import BaseModel

class Tool(BaseModel):
    name: str
    description: str

class Agent(BaseModel):
    name: str
    tools: list[Tool]

agent = Agent(name="ResearchBot", tools=[{"name": "search", "description": "Search"}])

d = agent.model_dump()
print("model_dump():", d)
print(type(d))

j = agent.model_dump_json()
print("model_dump_json():", j)
print(type(j))


### 6.8 — model_validate() and model_validate_json()

In [ ]:
from pydantic import BaseModel

class AgentConfig(BaseModel):
    model: str
    temperature: float
    max_tokens: int

# From dict (preferred in v2 over **unpacking)
raw_dict = {"model": "gpt-5", "temperature": "0.7", "max_tokens": "1000"}
cfg1 = AgentConfig.model_validate(raw_dict)
print(f"From dict: {cfg1}")

# From JSON string (LLM output, API response)
raw_json = '{"model": "gpt-5", "temperature": 0.7, "max_tokens": 1000}'
cfg2 = AgentConfig.model_validate_json(raw_json)
print(f"From JSON: {cfg2}")

### 6.9 — @field_validator

In [ ]:
from pydantic import BaseModel, Field, field_validator

class Tool(BaseModel):
    name: str = Field(min_length=3)
    description: str = Field(default="No description")
    max_retries: int = Field(ge=1, le=5)

    @field_validator("name")
    @classmethod
    def uppercase_name(cls, value: str) -> str:
        return value.upper()   # transform before storing

# Valid
tool = Tool(name="search", max_retries=3)
print(tool.name)   # SEARCH — transformed by validator

# Invalid retries
try:
    Tool(name="search", max_retries=10)
except Exception as e:
    print(f"Blocked: max_retries=10 exceeds le=5")

### 6.10 — @model_validator (cross-field validation)

In [ ]:
from pydantic import BaseModel, model_validator

class LLMConfig(BaseModel):
    model: str
    temperature: float
    use_json_mode: bool = False

    @model_validator(mode="after")
    def validate_o1_temperature(self):
        if self.model == "o1" and self.temperature != 1.0:
            raise ValueError(
                f"Model 'o1' only supports temperature=1.0, got {self.temperature}"
            )
        return self

# Valid
cfg1 = LLMConfig(model="gpt-5", temperature=0.7)
print(f"Valid: {cfg1}")

cfg2 = LLMConfig(model="o1", temperature=1.0)
print(f"Valid: {cfg2}")

# Invalid — o1 with non-1.0 temperature
try:
    LLMConfig(model="o1", temperature=0.7)
except Exception as e:
    print(f"Blocked: {e}")

### 6.11 — Complete agentic AI example

In [ ]:
from pydantic import BaseModel, Field, field_validator, model_validator
import json as _json

class Tool(BaseModel):
    name: str = Field(min_length=2)
    description: str
    enabled: bool = True

class AgentConfig(BaseModel):
    model: str = Field(default="gpt-4")
    temperature: float = Field(ge=0.0, le=2.0)
    max_tokens: int = Field(gt=0, le=4096)
    tools: list[Tool]
    system_prompt: str | None = None

    @field_validator("model")
    @classmethod
    def validate_model(cls, value: str) -> str:
        allowed = {"gpt-4", "gpt-5", "claude", "o1"}
        if value not in allowed:
            raise ValueError(f"Unsupported model: {value}")
        return value

    @model_validator(mode="after")
    def validate_tools_not_empty(self):
        active = [t for t in self.tools if t.enabled]
        if not active:
            raise ValueError("Agent needs at least one enabled tool")
        return self

raw = {
    "model": "gpt-5", "temperature": "0.7", "max_tokens": "2048",
    "tools": [
        {"name": "search", "description": "Search the web"},
        {"name": "calculator", "description": "Solve math", "enabled": False},
        {"name": "browser", "description": "Browse URLs"}
    ],
    "system_prompt": "You are a research assistant."
}

config = AgentConfig.model_validate(raw)
print("Loaded:", config.model)
print("Active tools:", [t.name for t in config.tools if t.enabled])
print("JSON output:")
print(config.model_dump_json(indent=2))


---

## Quick Reference

| Concept | Key Syntax | Core Use |
|---|---|---|
| **Decorator** | `@decorator` = `func = decorator(func)` | Add behavior without modifying function |
| **`functools.wraps`** | `@wraps(func)` inside wrapper | Preserve `__name__`, `__doc__` |
| **Decorator factory** | Extra nesting level | `@decorator(args)` |
| **Dataclass** | `@dataclass` | Auto `__init__`, `__repr__`, `__eq__` |
| **`field()`** | `field(default_factory=list)` | Mutable defaults, hide from repr |
| **`frozen=True`** | `@dataclass(frozen=True)` | Immutable + hashable |
| **Typing** | `str \| None`, `list[str]` | Type hints for tools and validators |
| **Enum** | `class X(Enum)` | Named constants, safe lookup |
| **`@property`** | Getter/setter/deleter | Attribute syntax + method power |
| **`__call__`** | `def __call__(self, ...)` | Stateful callable object |
| **BaseModel** | `class X(BaseModel)` | Runtime validation from type hints |
| **`Field()`** | `Field(ge=0, le=2)` | Constraints, defaults, aliases |
| **Nested models** | `tool: Tool` | Recursive validation |
| **`model_dump()`** | `.model_dump()` | Object → dict (serialize out) |
| **`model_validate()`** | `X.model_validate(raw)` | dict → object (parse in) |
| **`@field_validator`** | Single field custom logic | Transform or validate one field |
| **`@model_validator`** | Cross-field logic | Validate relationships between fields |
